<a href="https://colab.research.google.com/github/awfajri/artificial-intelligence/blob/main/soal_uts_no_1_auf_fajri_ramadhani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Soal UTS Kecerdasan Buatan no.1
<br>
Nama : Auf Fajri Ramadhani
<br>
NPM  : 2410631170059

import library

In [ ]:
import pandas as pd
import numpy as np

langkah A: membaca dataset

In [ ]:
df = pd.read_csv("adult.csv")

print("=== Preview Dataset ===")
print(df.head())
print(f"\nShape dataset : {df.shape}")
print(f"Kolom dataset : {df.columns.tolist()}")

=== Preview Dataset ===
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356       

langkah B: melakukan reprocessing data

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print("\n=== Distribusi Kelas Target (income) ===")
print(df['income'].value_counts())

print("\n=== Handling Missing Values ===")
for col in df.columns:
    if df[col].dtype == object:
        missing = (df[col] == '?').sum()
        if missing > 0:
            mode_val = df[col][df[col] != '?'].mode()[0]
            df[col]  = df[col].replace('?', mode_val)
            print(f"  '{col}': {missing} nilai '?' -> diganti '{mode_val}'")

features = [
    'age',
    'workclass',
    'education',
    'occupation',
    'relationship',
    'race',
    'sex',
    'hours.per.week',
    'native.country'
]
target = 'income'

df = df[features + [target]].copy()

numeric_features     = ['age', 'hours.per.week']
categorical_features = [f for f in features if f not in numeric_features]

print("\nPreprocessing selesai!")
print(f"Fitur numerik   : {numeric_features}")
print(f"Fitur kategorik : {categorical_features}")


=== Distribusi Kelas Target (income) ===
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

=== Handling Missing Values ===
  'workclass': 1836 nilai '?' -> diganti 'Private'
  'occupation': 1843 nilai '?' -> diganti 'Prof-specialty'
  'native.country': 583 nilai '?' -> diganti 'United-States'

Preprocessing selesai!
Fitur numerik   : ['age', 'hours.per.week']
Fitur kategorik : ['workclass', 'education', 'occupation', 'relationship', 'race', 'sex', 'native.country']


langkah C: Mengelompokkan data berdasarkan kelas (<=50K dan >50K)

In [ ]:
classes    = df[target].unique()
total_data = len(df)
class_data = {}

print("\n=== Pengelompokan Data per Kelas ===")
for c in classes:
    subset        = df[df[target] == c]
    class_data[c] = subset
    print(f"  Kelas '{c}' : {len(subset)} data")


=== Pengelompokan Data per Kelas ===
  Kelas '<=50K' : 24720 data
  Kelas '>50K' : 7841 data


langkah D: menghitung probabilitas prior dari masing-masing kelas

In [ ]:
prior = {}

print("\n=== Probabilitas Prior ===")
for c in classes:
    prior[c] = len(class_data[c]) / total_data
    print(f"  P({c}) = {len(class_data[c])}/{total_data} = {prior[c]:.4f}")


=== Probabilitas Prior ===
  P(<=50K) = 24720/32561 = 0.7592
  P(>50K) = 7841/32561 = 0.2408


langkah E: menghitung probabilitas likelihood untuk setiap fitur terhadap kelas

1. gaussian likelihood untuk fitur numerik

In [ ]:
gaussian_params = {}

print("\n=== Parameter Gaussian untuk Fitur Numerik ===")
for c in classes:
    gaussian_params[c] = {}
    print(f"\n  Kelas '{c}':")
    for feat in numeric_features:
        mean = class_data[c][feat].mean()
        std  = max(class_data[c][feat].std(), 1e-9)
        gaussian_params[c][feat] = (mean, std)
        print(f"    {feat}: mean = {mean:.2f}, std = {std:.2f}")


def gaussian_likelihood(x, mean, std):
    exponent = -((x - mean) ** 2) / (2 * std ** 2)
    return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(exponent)


=== Parameter Gaussian untuk Fitur Numerik ===

  Kelas '<=50K':
    age: mean = 36.78, std = 14.02
    hours.per.week: mean = 38.84, std = 12.32

  Kelas '>50K':
    age: mean = 44.25, std = 10.52
    hours.per.week: mean = 45.47, std = 11.01


2. categorical likelihood untuk fitur kategorik

In [ ]:
categorical_params = {}

print("\n=== Parameter Categorical Likelihood (dengan Laplace Smoothing) ===")
for c in classes:
    categorical_params[c] = {}
    for feat in categorical_features:
        value_counts   = class_data[c][feat].value_counts()
        total_in_class = len(class_data[c])
        unique_vals    = df[feat].nunique()
        prob_dict      = {}
        for val in df[feat].unique():
            count          = value_counts.get(val, 0)
            prob_dict[val] = (count + 1) / (total_in_class + unique_vals)
        categorical_params[c][feat] = prob_dict

print("  Categorical likelihood berhasil dihitung untuk semua fitur dan kelas.")



=== Parameter Categorical Likelihood (dengan Laplace Smoothing) ===
  Categorical likelihood berhasil dihitung untuk semua fitur dan kelas.


langkah F: mengalikan prior dan likelihood untuk mendapatkan probabilitas posterior

In [ ]:
def hitung_posterior(sample):
    log_posteriors = {}
    for c in classes:
        log_prob = np.log(prior[c])
        for feat in features:
            val = sample[feat]
            if feat in numeric_features:
                mean, std  = gaussian_params[c][feat]
                likelihood = max(gaussian_likelihood(val, mean, std), 1e-300)
            else:
                likelihood = categorical_params[c][feat].get(val, 1e-6)
            log_prob += np.log(likelihood)
        log_posteriors[c] = log_prob
    return log_posteriors


contoh_sample    = df.sample(1, random_state=7).iloc[0][features].to_dict()
contoh_posterior = hitung_posterior(contoh_sample)

print("\n=== Contoh Hasil Perhitungan Posterior (Langkah F) ===")
print(f"  log P(<=50K | X) = {contoh_posterior['<=50K']:.4f}")
print(f"  log P(>50K  | X) = {contoh_posterior['>50K']:.4f}")
print("  -> Posterior berhasil dihitung untuk kedua kelas.")


=== Contoh Hasil Perhitungan Posterior (Langkah F) ===
  log P(<=50K | X) = -24.4985
  log P(>50K  | X) = -33.8594
  -> Posterior berhasil dihitung untuk kedua kelas.


langkah G: Menentukan hasil prediksi berdasarkan nilai probabilitas terbesar

In [ ]:
def predict(sample):
    log_posteriors  = hitung_posterior(sample)
    predicted_class = max(log_posteriors, key=log_posteriors.get)
    return predicted_class, log_posteriors

langkah H: Menghitung Akurasi Sederhana dari Model yang Dibuat

In [ ]:
print("\n=== Menghitung Akurasi (500 sampel acak) ===")
test_df = df.sample(500, random_state=42)

correct = 0
for _, row in test_df.iterrows():
    sample = row[features].to_dict()
    pred, _ = predict(sample)
    if pred == row[target]:
        correct += 1

accuracy = correct / len(test_df)
print(f"Jumlah prediksi benar : {correct} / {len(test_df)}")
print(f"Akurasi Model         : {accuracy * 100:.2f}%")


=== Menghitung Akurasi (500 sampel acak) ===
Jumlah prediksi benar : 423 / 500
Akurasi Model         : 84.60%


pengujian model

In [ ]:
print("\n" + "=" * 60)
print("         UJI MODEL DENGAN SATU DATA INPUT")
print("=" * 60)

uji_input = {
    'age'           : 38,
    'workclass'     : 'Private',
    'education'     : 'Bachelors',
    'occupation'    : 'Exec-managerial',
    'relationship'  : 'Husband',
    'race'          : 'White',
    'sex'           : 'Male',
    'hours.per.week': 45,
    'native.country': 'United-States'
}

print("\nData Input:")
for k, v in uji_input.items():
    print(f"  {k:<20} : {v}")

hasil, log_post = predict(uji_input)

print("\n--- Log-Posterior per Kelas ---")
for c, lp in log_post.items():
    print(f"  log P({c} | X) = {lp:.4f}")

print(f"\n>>> PREDIKSI  : {hasil}")
if hasil == '>50K':
    print(">>> KESIMPULAN: Penghasilan TINGGI (> $50.000 / tahun)")
else:
    print(">>> KESIMPULAN: Penghasilan RENDAH (<= $50.000 / tahun)")


         UJI MODEL DENGAN SATU DATA INPUT

Data Input:
  age                  : 38
  workclass            : Private
  education            : Bachelors
  occupation           : Exec-managerial
  relationship         : Husband
  race                 : White
  sex                  : Male
  hours.per.week       : 45
  native.country       : United-States

--- Log-Posterior per Kelas ---
  log P(<=50K | X) = -14.1605
  log P(>50K | X) = -11.8750

>>> PREDIKSI  : >50K
>>> KESIMPULAN: Penghasilan TINGGI (> $50.000 / tahun)


Pertanyaan C: Analisis Sensitivitas — Ubah Jam Kerja per Minggu

In [ ]:
print("\n" + "=" * 60)
print("   ANALISIS: Efek Perubahan Jam Kerja per Minggu")
print("=" * 60)
print(f"\n{'Jam Kerja':<14} {'Prediksi':<12} {'Log P(<=50K)':<18} {'Log P(>50K)'}")
print("-" * 60)

for jam in [20, 30, 40, 45, 50, 60, 70, 80]:
    uji_temp = uji_input.copy()
    uji_temp['hours.per.week'] = jam
    pred, lp = predict(uji_temp)
    print(f"{jam:<14} {pred:<12} {lp['<=50K']:<18.4f} {lp['>50K']:.4f}")

print("\nKesimpulan Analisis:")
print("  Semakin banyak jam kerja -> P(>50K) cenderung naik.")
print("  Namun hasil prediksi akhir ditentukan oleh SEMUA fitur bersama,")
print("  bukan hanya jam kerja saja (itulah prinsip Naive Bayes).")


   ANALISIS: Efek Perubahan Jam Kerja per Minggu

Jam Kerja      Prediksi     Log P(<=50K)       Log P(>50K)
------------------------------------------------------------
20             >50K         -15.2049           -14.5491
30             >50K         -14.2929           -12.8611
40             >50K         -14.0399           -11.9976
45             >50K         -14.1605           -11.8750
50             >50K         -14.4458           -11.9586
60             >50K         -15.5106           -12.7441
70             >50K         -17.2344           -14.3541
80             >50K         -19.6172           -16.7886

Kesimpulan Analisis:
  Semakin banyak jam kerja -> P(>50K) cenderung naik.
  Namun hasil prediksi akhir ditentukan oleh SEMUA fitur bersama,
  bukan hanya jam kerja saja (itulah prinsip Naive Bayes).
